# 01 · Bronze ingestion
Load landing batches into the Bronze Delta table **as-is**, plus lineage columns
(`_ingest_ts`, `_source_file`, `_batch_id`). Bronze is append-only: no cleaning, no dedup.

Column names are snake_case by position because Delta does not allow spaces in names; values are untouched.

In [1]:
from pathlib import Path
from food_pipeline.spark_session import get_spark
from food_pipeline.bronze import ingest_batch

ROOT = Path.cwd().parent
spark = get_spark()
LANDING = (ROOT / "data" / "raw" / "landing").as_posix()
BRONZE = (ROOT / "lakehouse" / "bronze" / "survey_raw").as_posix()

## Ingest batch 1
Run this cell **once**. Append mode has no protection yet, so a second run would load the same 200 rows again (fixed in the next step: batch guard).

In [2]:
n = ingest_batch(spark, f"{LANDING}/batch_01.csv", "batch_01", BRONZE)
print("rows written:", n)

rows written: 200


## Check the Bronze table
Expect 200 rows, 17 string/timestamp columns, `_batch_id = batch_01`, and `feedback` still containing `"Negative "` (raw, untrimmed).

In [3]:
bronze = spark.read.format("delta").load(BRONZE)
print("bronze rows:", bronze.count())
bronze.printSchema()
bronze.groupBy("_batch_id").count().show()
bronze.select("age", "feedback", "raw_extra", "_ingest_ts", "_source_file", "_batch_id").show(3, truncate=False)

bronze rows: 200
root
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- monthly_income: string (nullable = true)
 |-- educational_qualifications: string (nullable = true)
 |-- family_size: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- pin_code: string (nullable = true)
 |-- output: string (nullable = true)
 |-- feedback: string (nullable = true)
 |-- raw_extra: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



+---------+-----+
|_batch_id|count|
+---------+-----+
| batch_01|  200|
+---------+-----+



+---+---------+---------+--------------------------+-----------------------------------------------------------------------------+---------+
|age|feedback |raw_extra|_ingest_ts                |_source_file                                                                 |_batch_id|
+---+---------+---------+--------------------------+-----------------------------------------------------------------------------+---------+
|20 |Positive |Yes      |2026-10-26 22:21:24.706446|file:///D:/My_Coding_projects/Pyspark%20project/data/raw/landing/batch_01.csv|batch_01 |
|24 |Positive |Yes      |2026-10-26 22:21:24.706446|file:///D:/My_Coding_projects/Pyspark%20project/data/raw/landing/batch_01.csv|batch_01 |
|22 |Negative |Yes      |2026-10-26 22:21:24.706446|file:///D:/My_Coding_projects/Pyspark%20project/data/raw/landing/batch_01.csv|batch_01 |
+---+---------+---------+--------------------------+-----------------------------------------------------------------------------+---------+
only showing 